In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

def is_checkout(path):
    return (path / 'train.py').is_file() and (path / 'src' / 'workrunner').is_dir()

cwd = Path.cwd()
ROOT = cwd if is_checkout(cwd) else next(path for path in sorted(cwd.iterdir()) if is_checkout(path))
sys.path.insert(0, str(ROOT / 'src'))
try:
    get_ipython().history_manager.enabled = False
    get_ipython().history_manager.db_log_output = False
except NameError:
    pass
print("Done importing")

Done importing


In [2]:
modules = {'numpy': 'numpy', 'torch': 'torch', 'pydantic': 'pydantic', 'loguru': 'loguru', 'cryptography': 'cryptography', 'ipykernel': 'ipykernel', 'Pillow': 'PIL', 'pyarrow': 'pyarrow'}
missing = [line for line in (ROOT / 'requirements.txt').read_text().splitlines() if importlib.util.find_spec(modules[line.split('>=')[0]]) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Ready')


Ready


In [3]:

CUSTOM_RELAY_LINK = ''

def check_relay(root, custom_link):
    from http.client import HTTPException
    from time import perf_counter
    from urllib.error import HTTPError

    import workrunner
    if Path(workrunner.__file__).resolve().parent != (root / 'src/workrunner').resolve():
        print('Restart the kernel before using this source checkout.')
        return
    from workrunner.config import ConfigError
    from workrunner.login_config import load_training_config
    from workrunner.paths import set_checkout_directories
    from workrunner.socks_proxy import http_opener
    from workrunner.vpn import ManagedVPN, VPNError

    set_checkout_directories(root)
    try:
        vpn = load_training_config(root / 'config.json')['vpn']
    except ConfigError as error:
        print(f'VPN check could not load config.json. {error}')
        return
    if not vpn['check_url']:
        print('Set vpn.check_url in config.json to run the VPN check.')
        return

    link = custom_link.strip()
    source = 'custom link' if link else 'configured link'
    try:
        with ManagedVPN(vpn, link=link or None) as proxy:
            opener = http_opener(proxy)
            started = perf_counter()
            with opener.open(vpn['check_url'], timeout=10) as response:
                response.read(256)
                ping_ms = (perf_counter() - started) * 1000
    except VPNError as error:
        print(f'VPN check failed ({source}). {error}')
    except HTTPError as error:
        print(f'VPN check failed ({source}). Health endpoint returned HTTP {error.code}.')
    except (OSError, HTTPException) as error:
        print(f'VPN check failed ({source}). Health request failed ({type(error).__name__}).')
    else:
        print(f'VPN check passed ({source}). HTTPS ping to health endpoint {ping_ms:.0f} ms.')

check_relay(ROOT, CUSTOM_RELAY_LINK)


VPN check passed (configured link). HTTPS ping to health endpoint 1745 ms.


In [4]:
import runpy

if 'trainer' in globals() and hasattr(trainer, 'stop_training'):
    trainer.stop_training()
    del trainer

def run_training(root):
    root = Path(root).resolve()
    workrunner = sys.modules.get('workrunner')
    if workrunner is not None and Path(workrunner.__file__).resolve().parent != root / 'src' / 'workrunner':
        print('Restart the kernel before using this source checkout.')
        return 1
    return runpy.run_path(str(root / 'train.py'))['run'](root)

exit_code = run_training(ROOT)


1 | job-0 | N/A | 00:01:52 | 0
1 | job-0 | 0% | 00:01:50 | 0
1 | job-0 | 6% | 00:01:49 | 26451451841.89
1 | job-0 | 58% | 00:01:48 | 360656015125.8
1 | job-0 | 58% | 00:01:47 | 640836674787.04
1 | job-0 | 68% | 00:01:46 | 879106487664.28
1 | job-0 | 68% | 00:01:45 | 1084145049199.9
1 | job-0 | 68% | 00:01:44 | 1262483549048.44
1 | job-0 | 68% | 00:01:43 | 1419036579627.44
1 | job-0 | 67% | 00:01:42 | 1557386534007.77
1 | job-0 | 67% | 00:01:42 | 0
